In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = (
    SparkSession.builder
    .appName("RetailPulseGoldPipeline")
    .config("spark.sql.warehouse.dir", "hdfs://namenode:8020/user/hive/warehouse")
    .config("hive.metastore.uris", "thrift://hive-metastore:9083")
    .config("spark.sql.sources.partitionOverwriteMode", "dynamic")
    .enableHiveSupport()
    .getOrCreate()
)

spark.sql("CREATE DATABASE IF NOT EXISTS gold")
gold_base = "hdfs://namenode:8020/user/hadoop/gold"

ev = spark.table("silver.retailpulse_events")
print("silver rows:", ev.count())
print("distinct event_id:", ev.select("event_id").distinct().count())   # لازم يساوي اللي فوقه
ev.groupBy("event_type").count().show()
ev.groupBy("is_late_arrival").count().show()

silver rows: 9409
distinct event_id: 9409
+------------------+-----+
|        event_type|count|
+------------------+-----+
|    product_viewed| 4826|
|      cart_updated| 1833|
|fulfillment_update|  710|
|  checkout_started| 1151|
|   order_confirmed|  889|
+------------------+-----+

+---------------+-----+
|is_late_arrival|count|
+---------------+-----+
|          false| 9409|
+---------------+-----+



In [2]:
chk = ev.groupBy("session_id").agg(
    countDistinct("channel").alias("n_channel"),
    countDistinct("store_id").alias("n_store"),
    countDistinct("customer_id").alias("n_customer"),
)
chk.filter((col("n_channel") > 1) | (col("n_store") > 1) | (col("n_customer") > 1)).count()

503

In [4]:
def first_ts(t):
    return min(when(col("event_type") == t, col("event_timestamp")))

session_df = (
    ev.groupBy("session_id")
    .agg(
        min("customer_id").alias("customer_id"),
        min("store_id").alias("store_id"),
        min("channel").alias("channel"),
        min("event_timestamp").alias("session_start_ts"),
        max("event_timestamp").alias("session_end_ts"),
        first_ts("product_viewed").alias("first_view_ts"),
        first_ts("cart_updated").alias("first_cart_ts"),
        first_ts("checkout_started").alias("first_checkout_ts"),
        first_ts("order_confirmed").alias("order_confirmed_ts"),
        max("order_id").alias("order_id"),
        count("*").alias("event_count"),
        max(col("is_late_arrival").cast("int")).alias("has_late_event"),
    )
    .withColumn("has_view", col("first_view_ts").isNotNull())
    .withColumn("has_cart", col("first_cart_ts").isNotNull())
    .withColumn("has_checkout", col("first_checkout_ts").isNotNull())
    .withColumn("has_order", col("order_confirmed_ts").isNotNull())
    .withColumn("session_date", to_date("session_start_ts"))
    .withColumn("date_key", date_format("session_start_ts", "yyyyMMdd").cast("int"))
    .withColumn("_loaded_at", current_timestamp())
)

(session_df.repartition(1, "session_date")
    .write.mode("overwrite").format("csv")
    .partitionBy("session_date")
    .option("path", f"{gold_base}/fact_digital_session")
    .saveAsTable("gold.fact_digital_session"))

In [6]:
fs = spark.table("gold.fact_digital_session")

funnel_df = (
    fs.groupBy("session_date", "date_key", "channel", "store_id")
    .agg(
        countDistinct("session_id").alias("sessions"),
        countDistinct(when(col("has_view"), col("session_id"))).alias("sessions_viewed"),
        countDistinct(when(col("has_cart"), col("session_id"))).alias("sessions_carted"),
        countDistinct(when(col("has_checkout"), col("session_id"))).alias("sessions_checkout"),
        countDistinct(when(col("has_order"), col("session_id"))).alias("sessions_ordered"),
    )
    .withColumn("view_to_cart_rate", round(col("sessions_carted") / col("sessions_viewed"), 4))
    .withColumn("cart_to_checkout_rate", round(col("sessions_checkout") / col("sessions_carted"), 4))
    .withColumn("checkout_to_order_rate", round(col("sessions_ordered") / col("sessions_checkout"), 4))
    .withColumn("overall_conversion_rate", round(col("sessions_ordered") / col("sessions"), 4))
    .withColumn("_loaded_at", current_timestamp())
)

(funnel_df.repartition(1, "session_date")
    .write.mode("overwrite").format("csv")
    .partitionBy("session_date")
    .option("path", f"{gold_base}/mart_digital_funnel_daily")
    .saveAsTable("gold.mart_digital_funnel_daily"))

In [7]:
accepted = spark.table("silver.retailpulse_events").count()
rej = (spark.table("silver.retailpulse_events_rejected")
       .groupBy("rejection_reason").count()
       .withColumnRenamed("rejection_reason", "outcome"))

acc_df = spark.createDataFrame([("accepted", accepted)], ["outcome", "count"])
total_bronze = spark.table("bronze.retailpulse_events").count()

run_id = spark.sql("SELECT date_format(current_timestamp(),'yyyyMMddHHmmss')").first()[0]

dq_df = (
    acc_df.unionByName(rej)
    .withColumnRenamed("count", "row_count")
    .withColumn("run_id", lit(run_id))
    .withColumn("run_ts", current_timestamp())
    .withColumn("source_table", lit("bronze.retailpulse_events"))
    .withColumn("total_source_rows", lit(total_bronze))
    .withColumn("pct_of_total", round(col("row_count") / lit(total_bronze) * 100, 2))
    .select("run_id", "run_ts", "source_table", "outcome",
            "row_count", "total_source_rows", "pct_of_total")
)

assert dq_df.agg(sum("row_count")).first()[0] == total_bronze, "reconciliation failed"

(dq_df.coalesce(1)
    .write.mode("append").format("csv")
    .option("path", f"{gold_base}/mart_data_quality_summary")
    .saveAsTable("gold.mart_data_quality_summary"))

In [8]:
spark.sql("""
SELECT channel, SUM(sessions_ordered)/SUM(sessions) AS conversion
FROM gold.mart_digital_funnel_daily
GROUP BY channel
""").show()

+-------+-------------------+
|channel|         conversion|
+-------+-------------------+
| mobile|0.10320901994796183|
|    web|0.09582942830365511|
+-------+-------------------+

